## Generación Facturas
### a.1 Creación de Scripts tarifario,generación de facturas y insert neo4j

In [1]:
# Instalación de librerías
!pip install neo4j openai pydantic pandas

import os

# ---------------------------------------------------------
# SCRIPT: GENERAR TARIFARIO
# ---------------------------------------------------------
with open("generar_tarifario.py", "w", encoding="utf-8") as f:
    f.write("""
import json

def generar_tarifario(output_file='tarifario_seguros.json', limit_marcas=None, años=None):
    marcas_config = {
        'Toyota': {'modelos': ['Corolla', 'Hilux', 'RAV4'], 'multi': 1.2},
        'Chevrolet': {'modelos': ['Onix', 'Silverado', 'Tracker'], 'multi': 1.0}
    }
    if años is None: años = [2024, 2025, 2026]

    items_base = [
        {'item': 'Puerta Delantera', 'cat': 'Repuesto', 'precio': 700},
        {'item': 'Espejo Retrovisor', 'cat': 'Repuesto', 'precio': 200},
        {'item': 'Cristal de Puerta', 'cat': 'Repuesto', 'precio': 140},
        {'item': 'Mano de Obra Pintura (por paño)', 'cat': 'Servicio', 'precio': 300},
        {'item': 'Ajuste Mecánico de Bisagras', 'cat': 'Servicio', 'precio': 120},
        {'item': 'Guardafango Delantero', 'cat': 'Repuesto', 'precio': 180},
        {'item': 'Parachoques Delantero', 'cat': 'Repuesto', 'precio': 450},
        {'item': 'Faros Delanteros (Par)', 'cat': 'Repuesto', 'precio': 600},
        {'item': 'Mano de Obra Enderezado', 'cat': 'Servicio', 'precio': 250},
        {'item': 'Kit de Grapas y Clips', 'cat': 'Repuesto', 'precio': 45},
        {'item': 'Líquido de Frenos', 'cat': 'Repuesto', 'precio': 15},
        {'item': 'Cambio de Aceite y Filtro', 'cat': 'Servicio', 'precio': 85},
        {'item': 'Revisión Sistema Eléctrico', 'cat': 'Servicio', 'precio': 60}
    ]

    tarifario = []
    for marca, cfg in marcas_config.items():
        if limit_marcas and marca not in list(marcas_config.keys())[:limit_marcas]: continue
        for mod in cfg['modelos']:
            for año in años:
                for base in items_base:
                    precio_final = round(base['precio'] * cfg['multi'] * (1 + (año-2020)*0.05), 2)
                    tarifario.append({
                        'marca': marca, 'modelo': mod, 'año': año,
                        'tipo_siniestro': 'Colisión Lateral', 'item': base['item'],
                        'categoria': base['cat'], 'precio_acordado': precio_final
                    })
    with open(output_file, 'w', encoding='utf-8') as fj:
        json.dump(tarifario, fj, indent=4)
    return tarifario
""")

# ---------------------------------------------------------
# SCRIPT: GENERAR FLUJO CON ESTRUCTURA XML EXACTA
# ---------------------------------------------------------
with open("generar_flujo_siniestros.py", "w", encoding="utf-8") as f:
    f.write("""
import json
import os
import xml.etree.ElementTree as ET
from xml.dom import minidom
from datetime import datetime
import random

def prettify(elem):
    rough_string = ET.tostring(elem, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    return reparsed.toprettyxml(indent='  ')

def generar_flujo_siniestros(n_ordenes=5, n_sobrecargo=1, n_duplicados=1, id_orden_ingreso=None, tarifario_file='tarifario_seguros.json', output_dir='facturas_emitidas'):
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    with open(tarifario_file, 'r', encoding='utf-8') as f: tarifario = json.load(f)

    res_ordenes = []
    n_normal = max(0, n_ordenes - n_sobrecargo - n_duplicados)
    tipos = (['normal'] * n_normal) + (['sobrecargo'] * n_sobrecargo) + (['duplicado'] * n_duplicados)
    last_normal_data = None

    for i, tipo in enumerate(tipos[:n_ordenes]):
        fac_id = f'FAC{i+1:03d}'
        # Seleccionar marca y modelo aleatorio del tarifario para cada factura
        v_pool = list(set([(t['marca'], t['modelo']) for t in tarifario]))
        marca, modelo = random.choice(v_pool)

        placa = f'ZSY-{7100+i}'
        # Cada factura tiene su propio OrderID con formato OI-2026-XXXX
        oi = id_orden_ingreso if (i == 0 and id_orden_ingreso) else f'OI-2026-{1001 + i}'

        items_base_pool = [t for t in tarifario if t['marca']==marca and t['modelo']==modelo]
        # Seleccionar aleatoriamente entre 3 y 6 items para que las facturas sean diferentes
        n_items_select = random.randint(3, min(7, len(items_base_pool)))
        items_base = random.sample(items_base_pool, n_items_select)

        items = []
        for it in items_base:
            p_unit = it['precio_acordado']
            if tipo == 'sobrecargo': p_unit = round(p_unit * 1.5, 2)
            items.append({**it, 'precio_final': p_unit})

        # Generar ítems duplicados en la misma factura (no en diferentes)
        if tipo == 'duplicado' and len(items) > 0:
            items.append(items[0].copy())

        total_sin = round(sum(it['precio_final'] for it in items), 2)
        iva = round(total_sin * 0.15, 2)
        total_con = round(total_sin + iva, 2)

        root = ET.Element('factura', id='comprobante', version='2.1.0')

        # infoTributaria
        it = ET.SubElement(root, 'infoTributaria')
        ET.SubElement(it, 'ambiente').text = '2'
        ET.SubElement(it, 'tipoEmision').text = '1'
        ET.SubElement(it, 'razonSocial').text = 'TALLER LOS PINOS S.A.'
        ET.SubElement(it, 'nombreComercial').text = 'TALLER LOS PINOS S.A.'
        ET.SubElement(it, 'ruc').text = '1792145632001'
        ET.SubElement(it, 'FactId').text = fac_id
        ET.SubElement(it, 'codDoc').text = '01'
        ET.SubElement(it, 'estab').text = '001'
        ET.SubElement(it, 'ptoEmi').text = '010'
        ET.SubElement(it, 'secuencial').text = str(111315001 + i)
        ET.SubElement(it, 'dirMatriz').text = 'AV. DE LOS GRANADOS'

        # infoFactura
        inf = ET.SubElement(root, 'infoFactura')
        ET.SubElement(inf, 'fechaEmision').text = datetime.now().strftime('%d/%m/%Y')
        ET.SubElement(inf, 'dirEstablecimiento').text = 'AV. DE LOS GRANADOS'
        ET.SubElement(inf, 'obligadoContabilidad').text = 'SI'
        ET.SubElement(inf, 'tipoIdentificacionComprador').text = '05'
        ET.SubElement(inf, 'razonSocialComprador').text = 'ASEGURADORA GENIAL S.A.'
        ET.SubElement(inf, 'identificacionComprador').text = '1790010020001'
        ET.SubElement(inf, 'direccionComprador').text = 'AV. AMAZONAS N32'
        ET.SubElement(inf, 'totalSinImpuestos').text = f'{total_sin:.2f}'
        ET.SubElement(inf, 'totalDescuento').text = '0.00'

        tci = ET.SubElement(inf, 'totalConImpuestos')
        timp = ET.SubElement(tci, 'totalImpuesto')
        ET.SubElement(timp, 'codigo').text = '2'
        ET.SubElement(timp, 'codigoPorcentaje').text = '4'
        ET.SubElement(timp, 'baseImponible').text = f'{total_sin:.2f}'
        ET.SubElement(timp, 'tarifa').text = '15'
        ET.SubElement(timp, 'valor').text = f'{iva:.2f}'

        ET.SubElement(inf, 'propina').text = '0.00'
        ET.SubElement(inf, 'importeTotal').text = f'{total_con:.2f}'
        ET.SubElement(inf, 'moneda').text = 'DOLAR'
        ET.SubElement(inf, 'placa').text = placa

        pagos = ET.SubElement(inf, 'pagos')
        pago = ET.SubElement(pagos, 'pago')
        ET.SubElement(pago, 'formaPago').text = '20'
        ET.SubElement(pago, 'total').text = f'{total_con:.2f}'

        # detalles
        dets = ET.SubElement(root, 'detalles')
        for itm in items:
            d = ET.SubElement(dets, 'detalle')
            ET.SubElement(d, 'codigoPrincipal').text = str(random.randint(1000, 9999))
            ET.SubElement(d, 'descripcion').text = itm['item']
            ET.SubElement(d, 'cantidad').text = '1.00000'
            ET.SubElement(d, 'precioUnitario').text = f'{itm["precio_final"]:.2f}'
            ET.SubElement(d, 'descuento').text = '0.00'
            ET.SubElement(d, 'precioTotalSinImpuesto').text = f'{itm["precio_final"]:.2f}'

            imps = ET.SubElement(d, 'impuestos')
            imp = ET.SubElement(imps, 'impuesto')
            ET.SubElement(imp, 'codigo').text = '2'
            ET.SubElement(imp, 'codigoPorcentaje').text = '4'
            ET.SubElement(imp, 'tarifa').text = '15'
            ET.SubElement(imp, 'baseImponible').text = f'{itm["precio_final"]:.2f}'
            ET.SubElement(imp, 'valor').text = f'{round(itm["precio_final"]*0.15, 2):.2f}'

        # infoAdicional
        iad = ET.SubElement(root, 'infoAdicional')
        ET.SubElement(iad, 'campoAdicional', nombre='OrderID').text = oi
        ET.SubElement(iad, 'campoAdicional', nombre='PLACA').text = placa
        ET.SubElement(iad, 'campoAdicional', nombre='MARCA').text = 'Toyota'
        ET.SubElement(iad, 'campoAdicional', nombre='MODELO').text = 'RAV4'

        with open(f'{output_dir}/Factura_{tipo}_{fac_id}.xml', 'w', encoding='utf-8') as fxml:
            fxml.write(prettify(root))

        res_ordenes.append({"FacId": fac_id, "OrderID": oi, "ID_ORDEN_INGRESO": oi, "placa": placa, "total": total_con, "tipo": tipo})

    with open('ordenes_reparacion.json', 'w', encoding='utf-8') as f_or:
        json.dump(res_ordenes, f_or, indent=4)
    return res_ordenes
""")

# ---------------------------------------------------------
# SCRIPT: CARGAR NEO4J
# ---------------------------------------------------------
with open("cargar_neo4j.py", "w", encoding="utf-8") as f:
  f.write ('''
import json
import os
import xml.etree.ElementTree as ET
from neo4j import GraphDatabase

def ingest_to_neo4j(uri, user, password, clear=True):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        if clear: session.run('MATCH (n) DETACH DELETE n')

        # Ingestar Tarifario
        with open('tarifario_seguros.json', 'r') as f: tar = json.load(f)
        session.run("""
            UNWIND $d AS r
            MERGE (v:Vehiculo {marca: r.marca, modelo: r.modelo})
            MERGE (i:TarifarioItem {nombre: r.item})
            MERGE (v)-[rel:PRECIO {siniestro: r.tipo_siniestro}]->(i)
            SET rel.precio = r.precio_acordado
        """, d=tar)

        # Ingestar Facturas desde XML
        for fn in os.listdir('facturas_emitidas'):
            if fn.endswith('.xml'):
                tree = ET.parse(f'facturas_emitidas/{fn}')
                root = tree.getroot()
                fid = root.find('infoTributaria/FactId').text
                oid = ""
                marca = ""
                modelo = ""
                for c in root.findall('infoAdicional/campoAdicional'):
                    if c.get('nombre') == 'OrderID': oid = c.text
                    if c.get('nombre') == 'MARCA': marca = c.text
                    if c.get('nombre') == 'MODELO': modelo = c.text

                session.run("""
                    MERGE (f:Factura {facId: $fid})
                    MERGE (o:OrdenIngreso {ordenId: $oid})
                    MERGE (f)-[:BASADA_EN]->(o)
                    WITH f
                    MATCH (v:Vehiculo {marca: $marca, modelo: $modelo})
                    MERGE (f)-[:PARA_VEHICULO]->(v)
                """, fid=fid, oid=oid, marca=marca, modelo=modelo)

                for d in root.findall('detalles/detalle'):
                    session.run("""
                        MATCH (f:Factura {facId: $fid})
                        CREATE (df:DetalleFactura {desc: $desc, p: $p})
                        CREATE (f)-[:CONTIENE]->(df)
                    """, fid=fid, desc=d.find('descripcion').text, p=float(d.find('precioUnitario').text))
    driver.close()
''')

print("Scripts generados con éxito.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 6.5 MB/s eta 0:00:00
Scripts generados con éxito.


###a.2 Generación de Tarifario y Facturas Parametrizadas
Asociamos las facturas generadas al `ID_ORDEN_INGRESO` extraído

In [2]:
import importlib
import pandas as pd
import generar_flujo_siniestros
import generar_tarifario
import shutil
importlib.reload(generar_flujo_siniestros)
importlib.reload(generar_tarifario)
from generar_tarifario import generar_tarifario
from generar_flujo_siniestros import generar_flujo_siniestros

# 1. Generar y mostrar Tarifario como DataFrame
tarifario_list = generar_tarifario(años=[2026])
df_tarifario = pd.DataFrame(tarifario_list)
print("\n--- Tarifario de Seguros (Vista Parcial) ---")
display(df_tarifario.head())

# 2. Generar Facturas vinculadas al ID_ORDEN_INGRESO
num_facturas = 100
ID_ORDEN_INGRESO='OI-2026-417' ## AL MENOS UNA ORDEN CREA CON LA ORDEN GENERADA ANTERIORMENTE, LUEGO REEMPLAZAR POR ID_ORDEN_INGRESO
if os.path.exists("facturas_emitidas"):
        shutil.rmtree("facturas_emitidas")
        os.makedirs("facturas_emitidas")
ordenes = generar_flujo_siniestros(n_ordenes=num_facturas, n_sobrecargo=30, n_duplicados=10, id_orden_ingreso=ID_ORDEN_INGRESO)
print(f"\nSe han generado {len(ordenes)} facturas XML en 'facturas_emitidas/'.")
# Mostrar facturas generadas en un DataFrame
df_facturas = pd.DataFrame(ordenes)
df_facturas['Archivo'] = df_facturas.apply(lambda x: f"Factura_{x['tipo']}_{x['FacId']}.xml", axis=1)
print("\n--- Resumen de Facturas Generadas ---")
display(df_facturas[['FacId', 'OrderID', 'tipo', 'Archivo', 'total']])



--- Tarifario de Seguros (Vista Parcial) ---


,marca,modelo,año,tipo_siniestro,item,categoria,precio_acordado
0,Toyota,Corolla,2026,Colisión Lateral,Puerta Delantera,Repuesto,1092.0
1,Toyota,Corolla,2026,Colisión Lateral,Espejo Retrovisor,Repuesto,312.0
2,Toyota,Corolla,2026,Colisión Lateral,Cristal de Puerta,Repuesto,218.4
3,Toyota,Corolla,2026,Colisión Lateral,Mano de Obra Pintura (por paño),Servicio,468.0
4,Toyota,Corolla,2026,Colisión Lateral,Ajuste Mecánico de Bisagras,Servicio,187.2



Se han generado 100 facturas XML en 'facturas_emitidas/'.

--- Resumen de Facturas Generadas ---


,FacId,OrderID,tipo,Archivo,total
0,FAC001,OI-2026-417,normal,Factura_normal_FAC001.xml,1004.64
1,FAC002,OI-2026-1002,normal,Factura_normal_FAC002.xml,2496.65
2,FAC003,OI-2026-1003,normal,Factura_normal_FAC003.xml,1067.43
3,FAC004,OI-2026-1004,normal,Factura_normal_FAC004.xml,1696.82
4,FAC005,OI-2026-1005,normal,Factura_normal_FAC005.xml,1372.41
...,...,...,...,...,...
95,FAC096,OI-2026-1096,duplicado,Factura_duplicado_FAC096.xml,2877.88
96,FAC097,OI-2026-1097,duplicado,Factura_duplicado_FAC097.xml,3109.60
97,FAC098,OI-2026-1098,duplicado,Factura_duplicado_FAC098.xml,1487.53
98,FAC099,OI-2026-1099,duplicado,Factura_duplicado_FAC099.xml,3004.95
